# 04 · QLoRA fine-tuning

**What this notebook decides.** The recipe for condition **C**, which differs
from **B** in exactly one respect: the frozen base model is held in 4-bit NF4
instead of bf16.

The comparison C vs B answers a practical question — *what does 4-bit
quantization cost in accuracy?* — and it only answers it if nothing else differs.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    environment_report,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

## 1 · The contrast must be clean

In [ ]:
from qwen_text2sql.config import load_config

lora = load_config(ROOT / "configs/qwen35_4b_lora.yaml")
qlora = load_config(ROOT / "configs/qwen35_4b_qlora.yaml")

assert qlora.quantization.enabled, "The QLoRA config does not enable quantization"
assert not lora.quantization.enabled, "The LoRA config unexpectedly enables quantization"

comparison = pd.DataFrame(
    {
        "LoRA (B)": {
            "model_id": lora.model.model_id,
            "rank": lora.lora.rank,
            "alpha": lora.lora.alpha,
            "learning_rate": lora.training.learning_rate,
            "epochs": lora.training.epochs,
            "effective_batch": lora.training.per_device_train_batch_size
            * lora.training.gradient_accumulation_steps,
            "seed": lora.training.seed,
            "4bit": lora.quantization.enabled,
        },
        "QLoRA (C)": {
            "model_id": qlora.model.model_id,
            "rank": qlora.lora.rank,
            "alpha": qlora.lora.alpha,
            "learning_rate": qlora.training.learning_rate,
            "epochs": qlora.training.epochs,
            "effective_batch": qlora.training.per_device_train_batch_size
            * qlora.training.gradient_accumulation_steps,
            "seed": qlora.training.seed,
            "4bit": qlora.quantization.enabled,
        },
    }
)
comparison["identical"] = comparison["LoRA (B)"] == comparison["QLoRA (C)"]
comparison

In [ ]:
confounds = comparison[~comparison["identical"]].index.drop("4bit", errors="ignore").tolist()
if confounds:
    print(f"CONFOUNDED: B and C also differ in {confounds}.")
    print("Any accuracy gap cannot be attributed to quantization alone.")
else:
    print("Clean contrast: B and C differ only in quantization.")

## 2 · What 4-bit actually buys

Base-weight memory is exact arithmetic once the parameter count is fixed. The
figures below are for the frozen base weights of a nominally 4-billion-parameter
model; they exclude optimizer state, activations and gradient checkpointing
buffers, which is why a run needs more memory than the table suggests.

In [ ]:
NOMINAL_PARAMS = 4_000_000_000

precisions = pd.DataFrame(
    [
        {"precision": "float32", "bits": 32},
        {"precision": "bfloat16 / float16", "bits": 16},
        {"precision": "int8", "bits": 8},
        {"precision": "NF4 (4-bit)", "bits": 4},
    ]
).assign(
    base_weights_gib=lambda frame: NOMINAL_PARAMS * frame["bits"] / 8 / 1024**3,
    relative_to_bf16=lambda frame: frame["bits"] / 16,
)
precisions.set_index("precision")

In [ ]:
quant = pd.Series(
    {
        "quant_type": qlora.quantization.quant_type,
        "double_quant": qlora.quantization.double_quant,
        "compute_dtype": qlora.quantization.compute_dtype,
    },
    name="value",
).to_frame()
print(
    "NF4 is a 4-bit datatype shaped for normally distributed weights; double\n"
    "quantization compresses the quantization constants themselves. Compute still\n"
    "happens in the compute_dtype below — 4-bit is a storage format, not an\n"
    "arithmetic one, so throughput does not improve proportionally.\n"
)
quant

## 3 · Running it

```bash
poetry run python scripts/train_adapter.py \
  --config configs/qwen35_4b_qlora.yaml \
  --train-data data/processed/bird_train.jsonl \
  --validation-data data/processed/bird_validation.jsonl
```

Requires CUDA and `bitsandbytes` (`poetry install --with quantization`). There is
no CPU fallback: if bitsandbytes is missing the run fails rather than silently
training in bf16, which would produce a mislabelled condition C.

## Reading this recipe

**What it shows.** That B and C differ only in quantization, and what the 4-bit
base costs in memory.

**What it does not show.** Whether 4-bit costs accuracy. That is a paired
comparison on identical examples, and it lives in
[`05_execution_evaluation`](05_execution_evaluation.ipynb). Expect the honest
answer to be "no detectable difference at this evaluation size" — which is a
result about the *power* of the comparison as much as about quantization, and
notebook [`00`](00_research_protocol.ipynb) quantifies that.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.